In [ ]:
import os

os.environ["KERAS_BACKEND"] = "jax"

import keras_hub

encoder = keras_hub.models.TextEmbedder.from_preset("all_minilm_l6_v2_en")

embedding = encoder.predict_on_batch(
    [
        "It's beautiful and sunny outside.",
        "The weather is very good today.",
        "It is raining.",
        "Oh wow, it is pouring",
    ]
)

In [14]:
print(embedding.shape)


(4, 384)


In [ ]:
# Calculate similarities between the embedding vecotrs
import jax.numpy as jnp


def cosine_similarity(x, y):
    """Compute cosine similarity between two vectors."""
    x = x / (jnp.linalg.norm(x) + 1e-8)
    y = y / (jnp.linalg.norm(y) + 1e-8)
    return jnp.dot(x, y)


print(cosine_similarity(embedding[0], embedding[1]))
print(cosine_similarity(embedding[0], embedding[2]))
print(cosine_similarity(embedding[0], embedding[3]))


0.6161209
0.43634173
0.29094055


In [21]:
weather_examples = {
    "rainy": [
        "It is raining outside.",
        "The weather is rainy.",
        "It's pouring outside.",
    ],
    "sunny": [
        "It is sunny outside.",
        "The weather is bright and sunny.",
        "It's a beautiful sunny day.",
    ],
}

mood_examples = {
    "active": [
        "I feel energetic.",
        "I want to do something active.",
        "I have lots of energy.",
    ],
    "chill": [
        "I want to relax.",
        "I feel like taking it easy.",
        "I want to do something calm.",
    ],
}

In [23]:
def make_prototype(examples):
    """Create a prototype embedding from several examples."""
    embeddings = encoder.predict_on_batch(examples)
    prototype = jnp.mean(embeddings, axis=0)
    return prototype / (jnp.linalg.norm(prototype) + 1e-8)


weather_prototypes = {
    name: make_prototype(examples) for name, examples in weather_examples.items()
}

mood_prototypes = {
    name: make_prototype(examples) for name, examples in mood_examples.items()
}

In [ ]:
def similarity_scores(embedding, prototypes):
    """Compare an embedding with each prototype."""
    return {
        name: float(cosine_similarity(embedding, prototype))
        for name, prototype in prototypes.items()
    }


x = encoder.predict_on_batch(["I'm full of energy today."])

print(similarity_scores(x[0], mood_prototypes))

{'active': 0.7348935604095459, 'chill': 0.42395609617233276}


In [30]:
def match_prototype(embedding, prototypes, threshold=0.5):
    """Return the closest prototype if similarity is high enough."""
    scores = similarity_scores(embedding, prototypes)

    best_label = max(scores, key=scores.get)
    best_score = scores[best_label]

    if best_score < threshold:
        return None

    return best_label

In [31]:
match_prototype(x[0], mood_prototypes)

'active'